# 06 · Clinical NLP with medspaCy

> **Synthetic data only.** Every dataset used in this notebook is synthetic (Synthea, seed 42) or research-use-only (NIH ChestX-ray14). There is no PHI. None of these models is clinically validated or cleared for patient care.

Sketch the clinical-note NLP pipeline: target-concept matching for problems/medications/allergies plus ConText negation. Notes here are synthetic and assumed de-identified upstream. Cells degrade gracefully if medspaCy is not installed.

> Notes reaching this pipeline are de-identified by the upstream `deid-service`. The output spans are redacted.

## A synthetic note
Hand-authored, no PHI.

In [ ]:
from __future__ import annotations
note = (
    'Patient with history of heart failure and type 2 diabetes. '
    'No evidence of pneumonia. Started on lisinopril. '
    'Allergic to penicillin.'
)
note

## Build a medspaCy pipeline
We add a few target rules and rely on ConText for negation. If medspaCy is unavailable we fall back to a simple keyword matcher so the notebook still illustrates the output shape.

In [ ]:
def run_medspacy(text):
    import medspacy
    from medspacy.target_matcher import TargetRule
    nlp = medspacy.load()
    rules = [
        TargetRule('heart failure', 'PROBLEM'),
        TargetRule('type 2 diabetes', 'PROBLEM'),
        TargetRule('pneumonia', 'PROBLEM'),
        TargetRule('lisinopril', 'MEDICATION'),
        TargetRule('penicillin', 'ALLERGY'),
    ]
    nlp.get_pipe('medspacy_target_matcher').add(rules)
    doc = nlp(text)
    return [(e.text, e.label_, bool(e._.is_negated)) for e in doc.ents]

def run_fallback(text):
    terms = {
        'heart failure': 'PROBLEM', 'type 2 diabetes': 'PROBLEM',
        'pneumonia': 'PROBLEM', 'lisinopril': 'MEDICATION',
        'penicillin': 'ALLERGY',
    }
    low = text.lower()
    out = []
    for term, label in terms.items():
        i = low.find(term)
        if i >= 0:
            negated = 'no evidence of ' + term in low or 'no ' + term in low
            out.append((term, label, negated))
    return out

## Extract entities
Problems, medications and the allergy are tagged; `pneumonia` is correctly marked negated by the ConText cue 'no evidence of'.

In [ ]:
try:
    entities = run_medspacy(note)
    engine = 'medspaCy'
except Exception:
    entities = run_fallback(note)
    engine = 'fallback keyword matcher'
import pandas as pd
print('engine:', engine)
pd.DataFrame(entities, columns=['span', 'label', 'negated'])

## Map to the serving contract
The serving API returns `NoteEntity(text_span_redacted, label, concept_code, negated)`. We redact spans to mirror that output shape.

In [ ]:
def to_contract(ents):
    return [
        {'text_span_redacted': '*' * len(span), 'label': label,
         'concept_code': None, 'negated': neg}
        for span, label, neg in ents
    ]
to_contract(entities)

### Notes
* Negation/assertion is the high-value, error-prone part — see the model card's failure modes.
* Next: evaluation & fairness (notebook 07).